Generate Generalized Questions for New Clusters Grouped by Rthema

In [4]:
%pip install mistralai
import os
import json
from pathlib import Path
from mistralai import Mistral
from time import sleep

  Using cached mistralai-1.8.2-py3-none-any.whl.metadata (33 kB)
  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.11.5-py3-none-any.whl.metadata (67 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached anyio-4.9.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached mistralai-1.8.2-py3-none-any.whl (374 kB)
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-p

In [5]:
#Mistral API Initialisng
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model = "open-mistral-nemo"
client = Mistral(api_key=api_key)

#System Prompt for MistralAI
system_prompt = """
You are a legal knowledge engineering expert.

Your task is to generate a **generalized competency question (GCQ)** from a list of **clustered German legal competency questions (CQs)**.

---

## Goal:
Create **one single abstract, representative legal question** that captures the core meaning of all input CQs.

## Instructions:
- Output one well-formed, precise, **legally relevant** question in German.
- The question should reflect the **shared semantics** of the cluster without copying specific details.
- Use **legal terminology** such as:
  - “Unter welchen Voraussetzungen…”
  - “Welche rechtliche Bedeutung hat…”
  - “Wer ist verpflichtet…”
- Output **only the question**, in natural, legal German.
"""

In [ ]:
#Generate Generalized CQs (GCQs) from clustered CQs
input_folder = Path("cluster_output_by_rthema")
output_path = Path("competency_questions_output/generalized_questions_all_clusters.json")
output = {}

for file in input_folder.glob("clusters_*.json"):
    thema = file.stem.replace("clusters_", "")
    with open(file, "r", encoding="utf-8") as f:
        clusters = json.load(f)

    output[thema] = {}

    for cluster_id, data in clusters.items():
        questions = data.get("questions", [])
        if not questions or len(questions) < 5:
            continue

        print(f"🔍 {thema} – {cluster_id} ({len(questions)} Fragen)")

        prompt = (
            system_prompt
            + "\n\nFragen:\n"
            + "\n".join(f"- {q}" for q in questions)
            + "\n\nGeneralisierte Frage:"
        )

        try:
            response = client.chat.complete(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ]
            )
            gcq = response.choices[0].message.content.strip()
            output[thema][cluster_id] = gcq

        except Exception as e:
            print(f"⚠️ Fehler bei {thema} - {cluster_id}: {e}")
            output[thema][cluster_id] = "ERROR"

        sleep(5)  # API Limit Handling

🔍 Wohnungseigentum – Cluster 39 (10 Fragen)
🔍 Wohnungseigentum – Cluster 68 (85 Fragen)
🔍 Wohnungseigentum – Cluster 14 (27 Fragen)
🔍 Wohnungseigentum – Cluster 36 (67 Fragen)
🔍 Wohnungseigentum – Cluster 35 (19 Fragen)
🔍 Wohnungseigentum – Cluster 30 (220 Fragen)
🔍 Wohnungseigentum – Cluster 23 (18 Fragen)
🔍 Wohnungseigentum – Cluster 66 (53 Fragen)
🔍 Wohnungseigentum – Cluster 61 (51 Fragen)
🔍 Wohnungseigentum – Cluster 65 (41 Fragen)
🔍 Wohnungseigentum – Cluster 59 (41 Fragen)
🔍 Wohnungseigentum – Cluster 10 (28 Fragen)
🔍 Wohnungseigentum – Cluster 2 (14 Fragen)
🔍 Wohnungseigentum – Cluster 5 (12 Fragen)
🔍 Wohnungseigentum – Cluster 8 (32 Fragen)
🔍 Wohnungseigentum – Cluster 7 (27 Fragen)
🔍 Wohnungseigentum – Cluster 29 (76 Fragen)
🔍 Wohnungseigentum – Cluster 9 (28 Fragen)
🔍 Wohnungseigentum – Cluster 12 (30 Fragen)
🔍 Wohnungseigentum – Cluster 40 (41 Fragen)
🔍 Wohnungseigentum – Cluster 63 (47 Fragen)
🔍 Wohnungseigentum – Cluster 49 (67 Fragen)
🔍 Wohnungseigentum – Cluster 50 (52 

In [ ]:
#Save the Output to a JSON File
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✅ Fertig – Generalized Questions gespeichert.")